## Challenge one: BigQuery: Feature Engineering for Fraud Detection

### Importing big query from Google Cloud library

In [2]:
from google.cloud import bigquery

### variables

In [9]:
import os

# --- Diagnostic: shows what each detection method returns ---
print("=== Project detection diagnostic ===")
print("env GOOGLE_CLOUD_PROJECT:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("env GCP_PROJECT         :", os.environ.get("GCP_PROJECT"))
try:
    import google.auth
    _creds, _proj = google.auth.default()
    print("google.auth project     :", _proj)
except Exception as e:
    print("google.auth failed      :", e)
print("=" * 36)


def detect_project_id():
    # 1. Explicit env var wins (lets anyone override without editing code)
    env = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if env:
        return env
    # 2. Ask Application Default Credentials what project we're running under
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    # 3. Last resort: query the metadata server (works on GCP runtimes)
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/project-id",
            headers={"Metadata-Flavor": "Google"},
        )
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except Exception:
        return None


class Config:
    def __init__(
        self,
        project_id=None,                      # None -> auto-detect from the runtime
        dataset="fraud",
        location="US",                        # must match the GCS bucket region
        source_uri="gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv",
        raw_table_name="fraud_data_raw",
        training_table_name="fraud_training_data",
    ):
        resolved = project_id or detect_project_id()
        if not resolved:
            raise ValueError(
                "Could not determine project_id. Pass it explicitly: "
                "Config(project_id='my-project')"
            )
        # bypass our own __setattr__ block during construction
        object.__setattr__(self, "project_id", resolved)
        object.__setattr__(self, "dataset", dataset)
        object.__setattr__(self, "location", location)
        object.__setattr__(self, "source_uri", source_uri)
        object.__setattr__(self, "raw_table_name", raw_table_name)
        object.__setattr__(self, "training_table_name", training_table_name)

    def __setattr__(self, name, value):
        raise AttributeError(f"Config is immutable; can't set {name!r}")

    @property
    def raw_table(self):
        return f"{self.project_id}.{self.dataset}.{self.raw_table_name}"

    @property
    def training_table(self):
        return f"{self.project_id}.{self.dataset}.{self.training_table_name}"


# --- Build the config ---
# Auto-detects on a GCP runtime. If detection fails, either:
#   - set os.environ["GOOGLE_CLOUD_PROJECT"] = "your-project" above, OR
#   - pass it directly: Config(project_id="your-project")
CFG = Config()

print("\nProject       :", CFG.project_id)
print("Raw table     :", CFG.raw_table)
print("Training table:", CFG.training_table)

=== Project detection diagnostic ===
env GOOGLE_CLOUD_PROJECT: qwiklabs-gcp-01-5fe45b5e4e14
env GCP_PROJECT         : None
google.auth project     : qwiklabs-gcp-01-5fe45b5e4e14

Project       : qwiklabs-gcp-01-5fe45b5e4e14
Raw table     : qwiklabs-gcp-01-5fe45b5e4e14.fraud.fraud_data_raw
Training table: qwiklabs-gcp-01-5fe45b5e4e14.fraud.fraud_training_data


### Logging

In [10]:
import logging, json, sys, uuid, datetime as dt

logger = logging.getLogger("fraud_fe")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

RUN_ID = str(uuid.uuid4())

def log_event(step: str, status: str, **kw):
    logger.info(json.dumps({"run_id": RUN_ID, "step": step, "status": status, **kw}))

log_event("init", "ok", started=dt.datetime.now(dt.timezone.utc).isoformat())

2026-06-02 15:16:16,779 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "init", "status": "ok", "started": "2026-06-02T15:16:16.779398+00:00"}


INFO:fraud_fe:{"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "init", "status": "ok", "started": "2026-06-02T15:16:16.779398+00:00"}


### Feature Engineering

In [11]:
class FraudFeatureEngineering:
    """Loads raw fraud data and engineers features into a training table."""

    FEATURE_SQL = """
    CREATE OR REPLACE TABLE `{training_table}` AS
    SELECT
      Applicant_ID,
      Age,
      Income,
      Number_of_Dependents,
      Amount_Requested,
      Application_Frequency_Last_Year,

      -- One-hot: Employment_Status
      CAST(Employment_Status = 'Employed'      AS INT64) AS Employment_Employed,
      CAST(Employment_Status = 'Self-Employed' AS INT64) AS Employment_Self_Employed,
      CAST(Employment_Status = 'Unemployed'    AS INT64) AS Employment_Unemployed,

      -- One-hot: Device_Type
      CAST(Device_Type = 'Desktop' AS INT64) AS Device_Desktop,
      CAST(Device_Type = 'Mobile'  AS INT64) AS Device_Mobile,
      CAST(Device_Type = 'Tablet'  AS INT64) AS Device_Tablet,

      -- One-hot: Age bins
      CAST(Age BETWEEN 18 AND 24 AS INT64) AS Age_18_24,
      CAST(Age BETWEEN 25 AND 34 AS INT64) AS Age_25_34,
      CAST(Age BETWEEN 35 AND 44 AS INT64) AS Age_35_44,
      CAST(Age BETWEEN 45 AND 54 AS INT64) AS Age_45_54,
      CAST(Age BETWEEN 55 AND 64 AS INT64) AS Age_55_64,
      CAST(Age >= 65            AS INT64) AS Age_65_plus,

      -- Income-to-Amount-Requested ratio
      SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,

      -- Time since previous assistance (days)
      DATE_DIFF(DATE(Application_Date), DATE(Previous_Assistance_Date), DAY) AS Time_Since_Previous_Assistance,
      COALESCE(DATE_DIFF(DATE(Application_Date), DATE(Previous_Assistance_Date), DAY), -1) AS Time_Since_Previous_Assistance_Filled,

      -- Booleans -> 0/1
      CAST(Previous_Assistance_Received AS INT64) AS Previous_Assistance_Received,
      CAST(Supporting_Doc_Verified      AS INT64) AS Supporting_Doc_Verified,
      CAST(Fraudulent                   AS INT64) AS Fraudulent
    FROM `{raw_table}`
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = bigquery.Client(project=cfg.project_id, location=cfg.location)
        self.rows_in = 0
        self.rows_out = 0

    def ensure_dataset(self):
        ds = bigquery.Dataset(f"{self.cfg.project_id}.{self.cfg.dataset}")
        ds.location = self.cfg.location
        self.client.create_dataset(ds, exists_ok=True)
        log_event("dataset_ready", "ok", dataset=self.cfg.dataset)

    def load_raw(self):
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        )
        job = self.client.load_table_from_uri(
            self.cfg.source_uri, self.cfg.raw_table, job_config=job_config
        )
        job.result()
        self.rows_in = self.client.get_table(self.cfg.raw_table).num_rows
        log_event("load_raw", "ok", rows=self.rows_in, job_id=job.job_id)

    def build_features(self):
        sql = self.FEATURE_SQL.format(
            training_table=self.cfg.training_table, raw_table=self.cfg.raw_table
        )
        job = self.client.query(sql)
        job.result()
        tbl = self.client.get_table(self.cfg.training_table)
        self.rows_out = tbl.num_rows
        log_event("build_features", "ok", rows=self.rows_out, cols=len(tbl.schema), job_id=job.job_id)

    def validate(self) -> bool:
        df = self.client.query(f"""
        SELECT
          COUNTIF(Employment_Employed + Employment_Self_Employed + Employment_Unemployed != 1) AS bad_employment,
          COUNTIF(Device_Desktop + Device_Mobile + Device_Tablet != 1) AS bad_device,
          COUNTIF(Age_18_24+Age_25_34+Age_35_44+Age_45_54+Age_55_64+Age_65_plus != 1) AS bad_age,
          COUNTIF(Fraudulent IS NULL) AS null_target
        FROM `{self.cfg.training_table}`
        """).to_dataframe().iloc[0]

        checks = {
            "row_count_preserved": self.rows_out == self.rows_in,
            "employment_onehot_exclusive": int(df.bad_employment) == 0,
            "device_onehot_exclusive": int(df.bad_device) == 0,
            "age_bins_exclusive": int(df.bad_age) == 0,
            "target_not_null": int(df.null_target) == 0,
        }
        for name, ok in checks.items():
            log_event("quality_check", "ok" if ok else "fail", check=name)
        return all(checks.values())

    def run(self):
        self.ensure_dataset()
        self.load_raw()
        self.build_features()
        passed = self.validate()
        log_event("run", "ok" if passed else "fail", rows_in=self.rows_in, rows_out=self.rows_out)
        return passed

### Running Feature

In [12]:
pipeline = FraudFeatureEngineering(CFG)
passed = pipeline.run()
print(f"\nDone — {pipeline.rows_out:,} rows in {CFG.training_table}")
print("Quality checks:", "PASSED" if passed else "FAILED")

2026-06-02 15:22:03,411 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "dataset_ready", "status": "ok", "dataset": "fraud"}


INFO:fraud_fe:{"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "dataset_ready", "status": "ok", "dataset": "fraud"}


2026-06-02 15:22:07,519 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "load_raw", "status": "ok", "rows": 50000, "job_id": "3e2dc8b9-c8b2-4915-a91a-af4f09a4f730"}


INFO:fraud_fe:{"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "load_raw", "status": "ok", "rows": 50000, "job_id": "3e2dc8b9-c8b2-4915-a91a-af4f09a4f730"}


2026-06-02 15:22:10,616 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "build_features", "status": "ok", "rows": 50000, "cols": 24, "job_id": "6410992a-4af1-4d7d-b5ee-a4682b403959"}


...

2026-06-02 15:22:13,318 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "quality_check", "status": "ok", "check": "target_not_null"}


INFO:fraud_fe:{"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "quality_check", "status": "ok", "check": "target_not_null"}


2026-06-02 15:22:13,319 | INFO | {"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "run", "status": "ok", "rows_in": 50000, "rows_out": 50000}


INFO:fraud_fe:{"run_id": "f210c6e3-53ee-4d43-bf15-d6701ea39831", "step": "run", "status": "ok", "rows_in": 50000, "rows_out": 50000}



Done — 50,000 rows in qwiklabs-gcp-01-5fe45b5e4e14.fraud.fraud_training_data
Quality checks: PASSED


### Results

In [14]:
pipeline.client.query(f"SELECT * FROM `{CFG.training_table}` LIMIT 50").to_dataframe()

,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Application_Frequency_Last_Year,Employment_Employed,Employment_Self_Employed,Employment_Unemployed,Device_Desktop,...,Age_35_44,Age_45_54,Age_55_64,Age_65_plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance,Time_Since_Previous_Assistance_Filled,Previous_Assistance_Received,Supporting_Doc_Verified,Fraudulent
0,1660,19,51701,2,2303,1,1,0,0,0,...,0,0,0,0,22.449414,<NA>,-1,0,0,0
1,2163,53,0,5,9030,1,0,1,0,1,...,0,1,0,0,0.000000,<NA>,-1,0,1,0
2,3275,47,59387,2,3922,1,0,0,1,0,...,0,1,0,0,15.142019,<NA>,-1,0,0,0
3,3560,61,40503,0,1722,1,0,0,1,0,...,0,0,1,0,23.520906,<NA>,-1,0,1,0
4,4071,32,0,1,4253,1,0,0,1,1,...,0,0,0,0,0.000000,<NA>,-1,0,1,0
5,4442,43,0,1,8748,1,0,1,0,1,...,1,0,0,0,0.000000,<NA>,-1,0,1,0
6,6254,62,0,5,5453,1,0,0,1,0,...,0,0,1,0,0.000000,<NA>,-1,0,1,0
7,6570,50,0,4,5048,1,1,0,0,1,...,0,1,0,0,0.000000,<NA>,-1,0,1,0
8,6729,36,67672,3,5126,1,0,1,0,1,...,1,0,0,0,13.201717,<NA>,-1,0,0,0
9,6976,62,0,5,3735,1,0,0,1,0,...,0,0,1,0,0.000000,<NA>,-1,0,1,0
